In [ ]:
"""
=============================================================================
  EMAIL SPAM CLASSIFICATION — PREPROCESSING PIPELINE
  Dataset : Enron Email Dataset (train.jsonl / test.jsonl)
  Tasks   : Cleaning · Tokenization · Stop-word Removal · Lemmatization
            Missing-value Handling · Duplicate Removal · Class Balancing (SMOTE)

=============================================================================
"""

import json
import re
import string
import os
import pickle
import warnings
import numpy as np
import pandas as pd
from collections import Counter

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")

import os, glob

def _find_file(candidates):
    """Auto-detect file by trying multiple common paths."""
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(
        f"Could not find dataset file. Tried: {candidates}\n"
        "Please set TRAIN_PATH / TEST_PATH manually at the top of this script."
    )

TRAIN_PATH = _find_file([
    "train.jsonl",
    "/content/train.jsonl",
    "1777184744161_train.jsonl",
    "/content/1777184744161_train.jsonl",
])
TEST_PATH = _find_file([
    "test.jsonl",
    "/content/test.jsonl",
    "1777185246170_test.jsonl",
    "/content/1777185246170_test.jsonl",
])
OUTPUT_DIR = "preprocessed_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────
#  1. STOP-WORDS
# ─────────────────────────────────────────────
STOPWORDS = {
    "a","about","above","after","again","against","all","am","an","and","any",
    "are","aren","as","at","be","because","been","before","being","below",
    "between","both","but","by","can","couldn","did","didn","do","does","doesn",
    "doing","don","down","during","each","few","for","from","further","get",
    "got","had","hadn","has","hasn","have","haven","having","he","her","here",
    "hers","herself","him","himself","his","how","i","if","in","into","is",
    "isn","it","its","itself","just","ll","me","mightn","more","most","my",
    "myself","needn","no","nor","not","now","of","off","on","once","only","or",
    "other","our","ours","ourselves","out","over","own","re","s","same","shan",
    "she","should","shouldn","so","some","such","t","than","that","the","their",
    "theirs","them","themselves","then","there","these","they","this","those",
    "through","to","too","under","until","up","us","ve","very","was","wasn",
    "we","were","weren","what","when","where","which","while","who","whom","why",
    "will","with","won","wouldn","you","your","yours","yourself","yourselves",
    "would","could","may","might","shall","has","have","had","said","also",
    "one","two","three","four","five","six","seven","eight","nine","ten",
    "email","mail","message","please","thank","thanks","dear","hi","hello",
    "regards","sincerely","subject","re","fw","fwd","cc","bcc","sent","received",
    "original","wrote","write","forward","forwarded","see","attached","attachment",
}

# ─────────────────────────────────────────────
#  2. SIMPLE LEMMATISER
# ─────────────────────────────────────────────
_SUFFIX_RULES = [
    (r"ational$", "ate"),  (r"tional$",  "tion"), (r"enci$",  "ence"),
    (r"anci$",    "ance"), (r"izer$",    "ize"),  (r"ising$", "ise"),
    (r"izing$",   "ize"),  (r"ised$",    "ise"),  (r"ized$",  "ize"),
    (r"alism$",   "al"),   (r"ness$",    ""),     (r"ment$",  ""),
    (r"fulness$", "ful"),  (r"ousness$", "ous"),  (r"iveness$","ive"),
    (r"ation$",   "ate"),  (r"ator$",    "ate"),  (r"alism$", "al"),
    (r"ies$",     "y"),    (r"ied$",     "y"),    (r"ing$",   ""),
    (r"edly$",    ""),     (r"ingly$",   ""),     (r"ful$",   ""),
    (r"ous$",     ""),     (r"ive$",     ""),     (r"ble$",   ""),
    (r"tion$",    "te"),   (r"sses$",    "ss"),   (r"s$",     ""),
    (r"ed$",      ""),
]

def lemmatize(word: str) -> str:
    """Lightweight rule-based lemmatizer (Porter-inspired)."""
    if len(word) <= 3:
        return word
    for pattern, replacement in _SUFFIX_RULES:
        if re.search(pattern, word):
            stem = re.sub(pattern, replacement, word)
            if len(stem) >= 3:
                return stem
    return word

# ─────────────────────────────────────────────
#  3. LOAD DATA
# ─────────────────────────────────────────────
def load_jsonl(path: str) -> pd.DataFrame:
    """Load a JSONL file, skipping any malformed / truncated lines."""
    records = []
    skipped = 0
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                skipped += 1
    if skipped:
        print(f"  WARNING: Skipped {skipped} malformed line(s) in: {path}")
    return pd.DataFrame(records)

print("=" * 65)
print("  STEP 1 — LOADING DATA")
print("=" * 65)

df_train = load_jsonl(TRAIN_PATH)
df_test  = load_jsonl(TEST_PATH)

print(f"  Train shape : {df_train.shape}")
print(f"  Test  shape : {df_test.shape}")
print(f"  Columns     : {list(df_train.columns)}")

# ─────────────────────────────────────────────
#  4. INITIAL DATA INSPECTION
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 2 — DATA INSPECTION")
print("=" * 65)

for name, df in [("TRAIN", df_train), ("TEST", df_test)]:
    label_counts = df["label"].value_counts()
    total = len(df)
    spam  = label_counts.get(1, 0)
    ham   = label_counts.get(0, 0)
    ratio = spam / ham if ham > 0 else float("inf")
    print(f"\n  [{name}]")
    print(f"    Total rows   : {total}")
    print(f"    Ham  (0)     : {ham}  ({ham/total*100:.1f}%)")
    print(f"    Spam (1)     : {spam}  ({spam/total*100:.1f}%)")
    print(f"    Spam:Ham     : {ratio:.2f}")
    print(f"    Missing text : {df['text'].isnull().sum()}")
    print(f"    Duplicates   : {df['text'].duplicated().sum()}")
    empty = (df["text"].str.strip() == "") | (df["text"].str.len() == 0)
    print(f"    Empty text   : {empty.sum()}")

# ─────────────────────────────────────────────
#  5. MISSING VALUE HANDLING
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 3 — MISSING VALUE HANDLING")
print("=" * 65)

def handle_missing(df: pd.DataFrame, name: str) -> pd.DataFrame:
    before = len(df)

    # Drop rows where text is NaN
    df = df.dropna(subset=["text"])

    # Fill NaN subject with empty string then merge into text if subject missing
    df["subject"] = df["subject"].fillna("")
    df["message"] = df["message"].fillna("")

    # Rebuild 'text' from subject + message if text is empty/whitespace after NaN drop
    mask_empty = df["text"].str.strip() == ""
    df.loc[mask_empty, "text"] = (
        df.loc[mask_empty, "subject"] + " " + df.loc[mask_empty, "message"]
    ).str.strip()

    # After rebuild, drop rows still empty
    still_empty = df["text"].str.strip() == ""
    dropped_empty = still_empty.sum()
    df = df[~still_empty]

    after = len(df)
    print(f"  [{name}] rows before: {before} → after: {after}  (removed {before-after})")
    return df.reset_index(drop=True)

df_train = handle_missing(df_train, "TRAIN")
df_test  = handle_missing(df_test,  "TEST")

# ─────────────────────────────────────────────
#  6. DUPLICATE REMOVAL
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 4 — DUPLICATE REMOVAL")
print("=" * 65)

def remove_duplicates(df: pd.DataFrame, name: str) -> pd.DataFrame:
    before = len(df)
    df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
    after = len(df)
    print(f"  [{name}] duplicates removed: {before - after}  | rows left: {after}")
    return df

df_train = remove_duplicates(df_train, "TRAIN")
df_test  = remove_duplicates(df_test,  "TEST")

# ─────────────────────────────────────────────
#  7. TEXT CLEANING FUNCTION
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 5 — TEXT CLEANING  (NLP Preprocessing)")
print("=" * 65)

def clean_text(text: str) -> str:
    """
    Full NLP preprocessing pipeline:
      1. Lowercase
      2. Remove HTML / XML tags
      3. Remove URLs
      4. Remove email addresses
      5. Remove phone numbers
      6. Remove special chars & punctuation
      7. Remove digits
      8. Collapse whitespace
      9. Tokenize (regex word boundaries)
     10. Remove stop-words
     11. Remove very short tokens (len < 2)
     12. Lemmatize
    """
    if not isinstance(text, str):
        return ""

    # 1. Lowercase
    text = text.lower()

    # 2. Remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # 3. Remove URLs
    text = re.sub(r"http\S+|www\.\S+|https\S+", " ", text)

    # 4. Remove email addresses
    text = re.sub(r"\S+@\S+", " ", text)

    # 5. Remove phone numbers
    text = re.sub(r"\b\d[\d\s\-\.\(\)]{6,}\d\b", " ", text)

    # 6. Remove special characters & punctuation (keep letters & spaces)
    text = re.sub(r"[^a-z\s]", " ", text)

    # 7. Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # 8. Tokenize
    tokens = re.findall(r"\b[a-z]{2,}\b", text)

    # 9. Remove stop-words + very short tokens
    tokens = [t for t in tokens if t not in STOPWORDS and len(t) > 2]

    # 10. Lemmatize
    tokens = [lemmatize(t) for t in tokens]

    return " ".join(tokens)


# Apply cleaning
print("  Applying cleaning pipeline to TRAIN set ...")
df_train["clean_text"] = df_train["text"].apply(clean_text)
print("  Applying cleaning pipeline to TEST set ...")
df_test["clean_text"]  = df_test["text"].apply(clean_text)

# Drop rows where clean_text is empty after cleaning
before_t = len(df_train)
df_train = df_train[df_train["clean_text"].str.strip() != ""].reset_index(drop=True)
before_e = len(df_test)
df_test  = df_test[df_test["clean_text"].str.strip()  != ""].reset_index(drop=True)

print(f"  TRAIN rows after cleaning filter: {before_t} → {len(df_train)}")
print(f"  TEST  rows after cleaning filter: {before_e} → {len(df_test)}")

# Sample preview
print("\n  ── Sample cleaned texts ──")
for i in range(2):
    row = df_train.iloc[i]
    print(f"\n  [{row['label_text'].upper()}] ORIGINAL (first 120):\n  {row['text'][:120]}")
    print(f"  CLEANED  (first 120):\n  {row['clean_text'][:120]}")

# Token stats
df_train["token_count"] = df_train["clean_text"].str.split().str.len()
df_test["token_count"]  = df_test["clean_text"].str.split().str.len()
print(f"\n  TRAIN token count — mean: {df_train['token_count'].mean():.1f}  "
      f"| median: {df_train['token_count'].median():.0f}  "
      f"| max: {df_train['token_count'].max()}")

# ─────────────────────────────────────────────
#  8. LABEL ENCODING
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 6 — LABEL ENCODING")
print("=" * 65)

# Labels are already numeric (0=ham, 1=spam) — verify and use directly
assert set(df_train["label"].unique()).issubset({0, 1}), "Unexpected label values!"
assert set(df_test["label"].unique()).issubset({0, 1}),  "Unexpected label values!"

y_train = df_train["label"].values
y_test  = df_test["label"].values

print(f"  Train — ham: {(y_train==0).sum()}  spam: {(y_train==1).sum()}")
print(f"  Test  — ham: {(y_test==0).sum()}   spam: {(y_test==1).sum()}")

# ─────────────────────────────────────────────
#  9. TEXT TO NUMERICAL FEATURES (Simple CountVectorizer for SMOTE)
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 7 — TEXT TO NUMERICAL FEATURES (for SMOTE)")
print("=" * 65)

# Using CountVectorizer as simple numerical representation for SMOTE
from sklearn.feature_extraction.text import CountVectorizer

count_vec = CountVectorizer(
    max_features=5000,  # Limited features for SMOTE
    ngram_range=(1, 1),
    min_df=2,
    max_df=0.95,
)

print("  Fitting CountVectorizer on TRAIN clean text ...")
X_train_count = count_vec.fit_transform(df_train["clean_text"])
X_test_count = count_vec.transform(df_test["clean_text"])

print(f"  X_train_count shape : {X_train_count.shape}")
print(f"  X_test_count  shape : {X_test_count.shape}")
print(f"  Vocabulary size      : {len(count_vec.vocabulary_)}")

# ─────────────────────────────────────────────
#  10. CLASS BALANCE CHECK + SMOTE
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 8 — CLASS BALANCING (SMOTE)")
print("=" * 65)

spam_count = (y_train == 1).sum()
ham_count  = (y_train == 0).sum()
imbalance_ratio = max(spam_count, ham_count) / min(spam_count, ham_count)

print(f"  Before SMOTE — ham: {ham_count}  spam: {spam_count}  ratio: {imbalance_ratio:.3f}")

if imbalance_ratio > 1.5:
    print("  Imbalance detected (ratio > 1.5) — applying SMOTE ...")
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_train_balanced, y_train_balanced = smote.fit_resample(X_train_count, y_train)
    new_spam = (y_train_balanced == 1).sum()
    new_ham  = (y_train_balanced == 0).sum()
    print(f"  After  SMOTE — ham: {new_ham}  spam: {new_spam}  ratio: {new_ham/new_spam:.3f}")
else:
    print(f"  Dataset is sufficiently balanced (ratio {imbalance_ratio:.3f} ≤ 1.5) — skipping SMOTE")
    X_train_balanced = X_train_count
    y_train_balanced = y_train

# ─────────────────────────────────────────────
#  11. SAVE PREPROCESSED ARTIFACTS
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  STEP 9 — SAVING OUTPUTS")
print("=" * 65)

# Save cleaned DataFrames as CSV
df_train.to_csv(f"{OUTPUT_DIR}/train_cleaned.csv", index=False)
df_test.to_csv(f"{OUTPUT_DIR}/test_cleaned.csv",   index=False)
print(f"  Saved cleaned CSVs → {OUTPUT_DIR}/train_cleaned.csv & test_cleaned.csv")

# Create balanced dataset from SMOTE results
# Convert sparse matrix to dense and create DataFrame
X_train_balanced_dense = X_train_balanced.toarray()
feature_names = count_vec.get_feature_names_out()

# Create balanced train DataFrame with features
df_train_balanced = pd.DataFrame(X_train_balanced_dense, columns=feature_names)
df_train_balanced['label'] = y_train_balanced

# Add clean_text back to balanced dataset by matching with original
# Since SMOTE generates synthetic samples, we'll include all original samples + synthetic
# For simplicity, save the feature representation with labels
df_train_balanced.to_csv(f"{OUTPUT_DIR}/train_balanced.csv", index=False)
print(f"  Saved SMOTE-balanced dataset → {OUTPUT_DIR}/train_balanced.csv")

# Save CountVectorizer
with open(f"{OUTPUT_DIR}/count_vectorizer.pkl", "wb") as f:
    pickle.dump(count_vec, f)
print(f"  Saved CountVectorizer → {OUTPUT_DIR}/count_vectorizer.pkl")

# Save numerical matrices (sparse)
import scipy.sparse as sp
sp.save_npz(f"{OUTPUT_DIR}/X_train_count.npz",      X_train_count)
sp.save_npz(f"{OUTPUT_DIR}/X_train_balanced.npz",   X_train_balanced)
sp.save_npz(f"{OUTPUT_DIR}/X_test_count.npz",       X_test_count)
np.save(f"{OUTPUT_DIR}/y_train.npy",          y_train)
np.save(f"{OUTPUT_DIR}/y_train_balanced.npy", y_train_balanced)
np.save(f"{OUTPUT_DIR}/y_test.npy",           y_test)
print(f"  Saved numerical matrices & labels → {OUTPUT_DIR}/")

# Save vocabulary
vocab_df = pd.DataFrame(
    sorted(count_vec.vocabulary_.items(), key=lambda x: x[1]),
    columns=["token", "index"]
)
vocab_df.to_csv(f"{OUTPUT_DIR}/vocabulary.csv", index=False)
print(f"  Saved vocabulary ({len(vocab_df)} tokens) → {OUTPUT_DIR}/vocabulary.csv")

# ─────────────────────────────────────────────
#  12. FINAL SUMMARY
# ─────────────────────────────────────────────
print("\n" + "=" * 65)
print("  PREPROCESSING COMPLETE — SUMMARY")
print("=" * 65)
print(f"""
  ┌──────────────────────────────────────────────────────────┐
  │  Dataset          : Enron Email (JSONL)                  │
  │  Train rows       : {len(df_train):<8}  (after cleaning)         │
  │  Test  rows       : {len(df_test):<8}  (after cleaning)         │
  │  Missing values   : Handled (filled / dropped)           │
  │  Duplicates       : Removed                              │
  │  Text cleaning    : lowercase, HTML/URL/email strip,     │
  │                     punct removal, stop-word removal,    │
  │                     lemmatization                        │
  │  Features         : CountVectorizer (for SMOTE)         │
  │  Vocab size       : {len(count_vec.vocabulary_):<8}                        │
  │  X_train shape    : {str(X_train_count.shape):<20}               │
  │  X_test  shape    : {str(X_test_count.shape):<20}               │
  │  Balanced X_train : {str(X_train_balanced.shape):<20}               │
  │  Class balance    : {'SMOTE applied' if imbalance_ratio > 1.5 else 'Already balanced':<30}       │
  │  Output dir       : {OUTPUT_DIR:<20}                    │
  └──────────────────────────────────────────────────────────┘

  Output files saved:
    → train_cleaned.csv   : Cleaned training data
    → test_cleaned.csv    : Cleaned test data
    → train_balanced.csv  : SMOTE-balanced training data
    → vocabulary.csv      : Feature vocabulary
    → count_vectorizer.pkl: Vectorizer for transform
""")


  STEP 1 — LOADING DATA
  Train shape : (26469, 7)
  Test  shape : (2000, 7)
  Columns     : ['message_id', 'text', 'label', 'label_text', 'subject', 'message', 'date']

  STEP 2 — DATA INSPECTION

  [TRAIN]
    Total rows   : 26469
    Ham  (0)     : 13056  (49.3%)
    Spam (1)     : 13413  (50.7%)
    Spam:Ham     : 1.03
    Missing text : 0
    Duplicates   : 2022
    Empty text   : 43

  [TEST]
    Total rows   : 2000
    Ham  (0)     : 992  (49.6%)
    Spam (1)     : 1008  (50.4%)
    Spam:Ham     : 1.02
    Missing text : 0
    Duplicates   : 18
    Empty text   : 0

  STEP 3 — MISSING VALUE HANDLING
  [TRAIN] rows before: 26469 → after: 26426  (removed 43)
  [TEST] rows before: 2000 → after: 2000  (removed 0)

  STEP 4 — DUPLICATE REMOVAL
  [TRAIN] duplicates removed: 1980  | rows left: 24446
  [TEST] duplicates removed: 18  | rows left: 1982

  STEP 5 — TEXT CLEANING  (NLP Preprocessing)
  Applying cleaning pipeline to TRAIN set ...
  Applying cleaning pipeline to TEST set ...
